# This is Pix2Pix GAN 
- This is made to convert grayscale manga panels to colored ones
- Project 1: 128x128 grayscale with 2000 images and 100 epochs
- Project 2: 128x128 grayscale with 2000 images and 200 epochs

*Results*
- project 1: 128x128 grayscale with 2000 images and 100 epochs with learning rate G: 2e-4 and D: 2e-4
    - failure
    - output is coloured image,
    - output image doesn't resemble the input one
    - output is completely different and unrecognisable
    - final D_loss: 0.2064, G_loss: 19.2775
    - Training Time: 66.2 mins

- Project 2: 128x128 grayscale with 2000 images and 200 epochs with learning rate G: 2e-4 and D: 2e-4
    - failure
    - output is colored image,
    - output image doesn't resemble the input one
    - output is completely different and unrecognisable
    - final D_loss: 0.104, G_loss: 20.9154
    - Training Time: 133 mins

- Project 3: 128x128 grayscale with 2000 images and 200 epochs with learning rate G: 2e-4 and D: 1e-5
    - failure
    - output is colored image
    - one thing to be noticed, i saw the speech cloud (maybe due to overfitting)
    - final D_loss: 0.9831 G_loss: 10.7815 (G loss was reduced significantly)
    = traning time: 139 mins

- Project 4: 128x128 grayscale with 2000 images and 150 epochs with learning rate G:2e-3 and D: 1e-5 
    - failure
    - early stopped because generator loss not converging
    - final [epoch 73/150] D_loss: 1.5403 G_loss: 13.8478

- Project 5, 6, 7 and 8:
    - failure
    - tweeks in the hyper parameters

- Project 9, improved structure of the generator with 200 epochs:
    - failure (maybe due to overfitting)
    - while training the outputs were really promising
    - in the training outputs, the rough structure of the original was seen
    - the generator was changed
        - 4 down layer
        - one bottleneck
        - 4 up layer (with skip connections)
        - 1 final layer
    - the problem might be smaller learning rate of discriminator (1e-5) as compare to generator (2e-4) 
    - and this problem might cause the discriminator fail to learn properly and the generator might get overpowerered.

- Project 10: 128x128 grayscale with 3000 images and 100 epochs with learning rate G: 2e-4 and D: 2e-4
    - success but need a lot of improvement
    - while training the output of the first epoch was similar to the black and white input with perfect shape of the original (color and texture weren't present)
    - the last epoch: the training  result of the generator was exact same copy of the original colored set, meaning the generator performed really well
    - at the time of inference, the shape and structure of the origin was same, but the color and texture weren't satisfactory
    - introduced 'gradient penelty' alongwith wasseterian loss (this is the reason of success).


- Project 11: 128x128 grayscale with 3000 images and 100 epochs with lambda_l1 = 20
    - improved results as compare to project 10

- Project 12: 128x128 grayscale with 3000 images and 100 epochs with lambda_l1 = 8
    - same

- Project 13: 128x128 grayscale with 9712 images and 100 epochs all parameters set to default
    - failure
    - serious overfitting
    - plus training time was too high

- Project 14: 128x128 grayscale with 1992 images
    - awesome, but still colors are dull
    - also working on one piece images (test from one piece manga)

- Project 15: 128x128 grayscale with 6722 images (from different mangas):
    -success
    

In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image
import os
from torch import autograd

*Hyperparameters*

In [ ]:

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 100
BATCH_SIZE = 8
# LR = 2e-4
LR_G=2e-4
LR_D=2e-4
LAMBDA_L1 = 20
curr_epoch=0
lambda_gp = 10  
print("CUDA Available:", torch.cuda.is_available())


print("Number of GPUs:", torch.cuda.device_count())


print("Current GPU Device:", torch.cuda.current_device())


print("GPU Name:", torch.cuda.get_device_name(torch.cuda.current_device()))


CUDA Available: True
Number of GPUs: 1
Current GPU Device: 0
GPU Name: NVIDIA GeForce GTX 1650


*The Generator*

In [ ]:
import torch
import torch.nn as nn

class Generator(nn.Module):
    def __init__(self):
        super().__init__()

        def down(in_c, out_c, norm=True):
            layers = [nn.Conv2d(in_c, out_c, 4, 2, 1)]
            if norm: layers.append(nn.BatchNorm2d(out_c))
            layers.append(nn.LeakyReLU(0.2))
            return layers

        def up(in_c, out_c, dropout=False):
            layers = [nn.ConvTranspose2d(in_c, out_c, 4, 2, 1),
                      nn.BatchNorm2d(out_c),
                      nn.ReLU()]
            if dropout: layers.append(nn.Dropout(0.5))
            return layers

        def final(in_c,out_c):
            layers=[nn.Conv2d(in_c,out_c,1,1,0),nn.Tanh()]
            return layers
        
        
        # Encoder (downsampling)
        self.down1 = nn.Sequential(*down(1, 64, norm=False))
        self.down2 = nn.Sequential(*down(64, 128))
        self.down3 = nn.Sequential(*down(128, 256))
        self.down4 = nn.Sequential(*down(256, 512))

        #bottleneck:
        self.down5=nn.Sequential(*down(512,512))


        # Decoder (upsampling with skip connections)
        
        self.up1 = nn.Sequential(*up(512,512, dropout=True))
        self.up2 = nn.Sequential(*up(768, 256, dropout=True))  
        self.up3 = nn.Sequential(*up(384, 128, dropout=True))  
        self.up4 = nn.Sequential(*up(192, 64))                 
        
        self.output = nn.Sequential(*final(64, 3))

    def forward(self, x):
        # Encoder
        d1 = self.down1(x)      
        d2 = self.down2(d1)     
        d3 = self.down3(d2)     
        d4 = self.down4(d3)    
        
        
        u1 = self.up1(d4)
        
        
        u2 = self.up2(torch.cat([u1, d3],dim=1))      
        
        u3 = self.up3(torch.cat([u2, d2],dim=1))      
        u4 = self.up4(torch.cat([u3,d1],dim=1)) 
        
        

        # Final output: map to 3 channels (RGB)
        out = self.output(u4) 
        return out


*The Discriminator*

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        def block(in_c, out_c, norm=True):
            layers = [nn.Conv2d(in_c, out_c, 4, 2, 1)]
            if norm: layers.append(nn.BatchNorm2d(out_c))
            layers.append(nn.LeakyReLU(0.2))
            return layers

        self.model = nn.Sequential(
            *block(4, 64, norm=False),  
            *block(64, 128),
            *block(128, 256),
            *block(256, 512),
            nn.Conv2d(512, 1, 4, 1, 1)
        )

    def forward(self, x, y):
        
        if x.shape[-2:] != y.shape[-2:]:
            y = F.interpolate(y, size=x.shape[-2:], mode='bilinear', align_corners=False)
        return self.model(torch.cat([x, y], dim=1))


*The Training Script*

In [ ]:


class GrayscaleToColorDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root = root_dir
        self.transform = transform
        self.files = os.listdir(os.path.join(root_dir, "gray"))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx): 
        gray_path = os.path.join(self.root, "gray", self.files[idx])
        color_path = os.path.join(self.root, "color", self.files[idx])

        gray = Image.open(gray_path).convert("L")  # 1 channel
        color = Image.open(color_path).convert("RGB")  # 3 channels

        if self.transform:
            gray = self.transform(gray)
            color = self.transform(color)

        return gray, color








In [20]:
# Transforms
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

In [21]:
# Dataset & Dataloader
dataset = GrayscaleToColorDataset("general_dataset", transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

In [22]:
test_dataset = GrayscaleToColorDataset("test_set", transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [23]:

# Models
gen = Generator().to(DEVICE)
disc = Discriminator().to(DEVICE)

# Losses & Optimizers
bce = nn.BCEWithLogitsLoss()
l1 = nn.L1Loss()
opt_gen = optim.Adam(gen.parameters(), lr=LR_G, betas=(0.5, 0.999))
opt_disc = optim.Adam(disc.parameters(), lr=LR_D, betas=(0.5, 0.999))

In [ ]:
import torch
from torchvision.utils import save_image
import os
def run_test_and_log(
    gen, disc, test_loader, device, log_path,
    lambda_l1=100, lambda_adv=1,  
    vgg_loss_fn=None, lambda_perc=0,
    total_var_loss_fn=None, lambda_tv=0
):
    gen.eval()
    total_g_loss, total_l1, total_adv, total_perc, total_tv = 0.0, 0.0, 0.0, 0.0, 0.0
    count = 0
    
    with torch.no_grad():
        with open(log_path, "w") as logf:
            for i, (gray, color) in enumerate(test_loader):
                gray, color = gray.to(device), color.to(device)
                fake = gen(gray)

                l1_loss = torch.mean(torch.abs(fake - color)) * lambda_l1
                adv_loss = -torch.mean(disc(gray, fake)) * lambda_adv
                perc_loss = (
                    vgg_loss_fn(fake, color) * lambda_perc if vgg_loss_fn else 0.0
                )
                tv_loss = (
                    total_var_loss_fn(fake) * lambda_tv if total_var_loss_fn else 0.0
                )

                g_loss = adv_loss + l1_loss + perc_loss + tv_loss

                total_g_loss += g_loss.item()
                total_l1 += l1_loss.item()
                total_adv += adv_loss.item()
                if perc_loss != 0.0:
                    total_perc += float(perc_loss)
                if tv_loss != 0.0:
                    total_tv += float(tv_loss)
                count += 1

                if i < 5:
                    save_image(fake, f"test_outputs/fake_{i}.png")
                    save_image(color, f"test_outputs/real_{i}.png")
                logf.write(
                    f"Batch {i}: Total G Loss = {g_loss.item():.4f}, "
                    f"L1 = {l1_loss.item():.4f}, Adv = {adv_loss.item():.4f}"
                )
                if vgg_loss_fn:
                    logf.write(f", Perc = {float(perc_loss):.4f}")
                if total_var_loss_fn:
                    logf.write(f", TV = {float(tv_loss):.4f}")
                logf.write("\n")
            avg_g_loss = total_g_loss / count if count > 0 else float('nan')
            avg_l1 = total_l1 / count if count > 0 else float('nan')
            avg_adv = total_adv / count if count > 0 else float('nan')
            avg_perc = total_perc / count if count > 0 else float('nan')
            avg_tv = total_tv / count if count > 0 else float('nan')

            logf.write(
                f"\nAverage G Loss: {avg_g_loss:.4f}, L1: {avg_l1:.4f}, Adv: {avg_adv:.4f}"
            )
            if vgg_loss_fn:
                logf.write(f", Perc: {avg_perc:.4f}")
            if total_var_loss_fn:
                logf.write(f", TV: {avg_tv:.4f}")
            logf.write("\n")
    gen.train()


In [ ]:


def gradient_penalty(disc, real, fake, gray, device="cuda"):
    batch_size, c, h, w = real.shape
    
    alpha = torch.rand(batch_size, 1, 1, 1).to(device)
    interpolated = (alpha * real + (1 - alpha) * fake).requires_grad_(True)

    
    mixed_scores = disc(gray, interpolated)

    
    grad_outputs = torch.ones_like(mixed_scores).to(device)
    gradients = autograd.grad(
        outputs=mixed_scores,
        inputs=interpolated,
        grad_outputs=grad_outputs,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]  

    
    gradients = gradients.view(gradients.shape[0], -1)
    
    gradient_norm = gradients.norm(2, dim=1)
    
    penalty = torch.mean((gradient_norm - 1) ** 2)
    return penalty


In [27]:
# opt_gen = optim.Adam(gen.parameters(), lr=LR_G, betas=(0.5, 0.999))
# opt_disc = optim.Adam(disc.parameters(), lr=LR_D, betas=(0.5, 0.999))

# # 2. Load the checkpoint
# checkpoint = torch.load('checkpoints/checkpoint49.pth', weights_only=True)

# # 3. Load state dictionaries
# gen.load_state_dict(checkpoint['model_state_dict'])
# disc.load_state_dict(checkpoint['disc_state_dict'])
# opt_gen.load_state_dict(checkpoint['opt_gen_state_dict'])
# opt_disc.load_state_dict(checkpoint['opt_disc_state_dict'])
# curr_epoch = checkpoint['epoch']  # Optional: use to resume from the correct epoch

# # # 4. Set models to training mode
# gen.train()
# disc.train()



In [ ]:

# Training loop

for epoch in range(curr_epoch,EPOCHS):
    for i, (gray, color) in enumerate(loader):
        gray, color = gray.to(DEVICE), color.to(DEVICE)
        # save_image(gray,f"i.png")
        # save_image(color,f"iw.png")
        # Generate fake color image
        fake = gen(gray)
        # print(fake[0][0])
        # print(fake.detach())
        fake = gen(gray)

        real_pred = disc(gray, color)
        fake_pred = disc(gray, fake.detach())
        gp = gradient_penalty(disc, color, fake.detach(), gray, device=DEVICE)

        d_loss = -(torch.mean(real_pred) - torch.mean(fake_pred)) + lambda_gp * gp

        opt_disc.zero_grad()
        d_loss.backward()
        opt_disc.step()

        # Generator training
        fake_pred = disc(gray, fake)
        g_adv =  -torch.mean(fake_pred)
        g_l1 = l1(fake, color) * LAMBDA_L1
        g_loss = g_adv + g_l1
        opt_gen.zero_grad()
        g_loss.backward()
        opt_gen.step()

        if i % 100 == 0 and i!=0:
            print(f"Epoch [{epoch}/{EPOCHS}] Step [{i}] D_loss: {d_loss.item():.4f} G_loss: {g_loss.item():.4f}")
            save_image(fake[:], f"output/fake/{epoch}_{i}.png")
            save_image(color[:], f"output/real/{epoch}_{i}.png") 

    if (epoch + 1) % 5 == 0:
        checkpoint = {
        'epoch': curr_epoch,          
        'model_state_dict': gen.state_dict(),
        'disc_state_dict': disc.state_dict(),
        'opt_gen_state_dict': opt_gen.state_dict(),
        'opt_disc_state_dict': opt_disc.state_dict(),
        
        }
        torch.save(checkpoint, f'checkpoints/checkpoint{curr_epoch}.pth')
        log_path = f"logs/test_{epoch+1}.log"
        run_test_and_log(gen,disc, test_loader, DEVICE, log_path)
    curr_epoch+=1
 

Epoch [0/100] Step [100] D_loss: 0.0529 G_loss: 3.4159
Epoch [0/100] Step [200] D_loss: -0.0358 G_loss: 2.3081
Epoch [0/100] Step [300] D_loss: -0.0039 G_loss: 1.8215
Epoch [0/100] Step [400] D_loss: -0.0085 G_loss: 1.3962
Epoch [0/100] Step [500] D_loss: 0.0485 G_loss: 1.3121
Epoch [0/100] Step [600] D_loss: -0.0132 G_loss: 1.7995
Epoch [0/100] Step [700] D_loss: 0.1776 G_loss: 1.4488
Epoch [0/100] Step [800] D_loss: -0.0266 G_loss: 1.4850
Epoch [1/100] Step [100] D_loss: 0.0386 G_loss: 1.9829
Epoch [1/100] Step [200] D_loss: -0.0515 G_loss: 1.7000
Epoch [1/100] Step [300] D_loss: -0.0654 G_loss: 1.8495
Epoch [1/100] Step [400] D_loss: -0.1341 G_loss: 2.2191
Epoch [1/100] Step [500] D_loss: -0.0818 G_loss: 2.1703
Epoch [1/100] Step [600] D_loss: -0.0844 G_loss: 2.3533
Epoch [1/100] Step [700] D_loss: -0.1263 G_loss: 2.7612
Epoch [1/100] Step [800] D_loss: -0.0654 G_loss: 2.0323
Epoch [2/100] Step [100] D_loss: -0.0365 G_loss: 2.1704
Epoch [2/100] Step [200] D_loss: -0.0815 G_loss: 2.3

KeyboardInterrupt: 

In [ ]:
checkpoint = {
    'epoch': curr_epoch,           
    'model_state_dict': gen.state_dict(),
    'disc_state_dict': disc.state_dict(),
    'opt_gen_state_dict': opt_gen.state_dict(),
    'opt_disc_state_dict': opt_disc.state_dict(),
    
}
torch.save(checkpoint, 'checkpoint2.pth')


In [ ]:
from torchvision import transforms
from PIL import Image

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])  
])

img = Image.open("donwload.jpg")
input_tensor = transform(img).unsqueeze(0).to(DEVICE)  
with torch.no_grad():
    gen.eval()
    output = gen(input_tensor)  
from torchvision.utils import save_image


output = (output + 1) / 2


save_image(output, "donwload.png")

